## Generators & Augmentation testing

In this notebook we will try new augmentation algorithm over the cells

In [1]:
import pandas as pd
from composition.cells import CELLS

hungry = set(pd.read_parquet("data/composition/cell_order_sheet.parquet")["floor"])
audit = pd.DataFrame([
    {"cell": c.name, "hungry": c.name in hungry, "looks_like": c.looks_like}
    for c in CELLS
]).assign(written=lambda d: d["looks_like"].notna())

print(f"{audit['written'].sum()} of {len(audit)} cells have looks_like")
print(f"hungry cells still missing one: {(audit['hungry'] & ~audit['written']).sum()}")

audit[~audit["written"]].sort_values("hungry", ascending=False)[["cell", "hungry"]]

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


44 of 44 cells have looks_like
hungry cells still missing one: 0


,cell,hungry


In [5]:
audit[audit["written"]]

,cell,hungry,looks_like,written
17,symbol_pile_no_grammar,True,Two or more bare technical tokens sitting side...,True


In [1]:
import pandas as pd

from augmentation.config import AugmentationPaths
from augmentation.loop import AugmentationLoop
from augmentation.parents import ParentPool
from dataset_registry import DATASETS


/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
paths = AugmentationPaths()
catalog = pd.read_parquet(paths.catalog).astype({"query_id": str})
selection = pd.read_parquet(paths.data_dir / "composition/cell_selection.parquet").astype({"query_id": str})
parents = ParentPool(catalog, selection, {d.name: d for d in DATASETS})

In [5]:
loop = AugmentationLoop(selection, sheet_path=paths.cell_order_sheet, parents=parents)
loop.plans(parents.available())

,cell,missing,parents,mints,unsatisfied,partial,action
0,legal_citation_canonical,400.0,350,"inject, stat_rewrite",,False,augment
1,datetime_token_present,399.0,193,"inject, stat_rewrite",,False,augment
2,single_token_char_blob,398.0,133143,stat_rewrite,,False,augment
3,business_temporal_reference,398.0,9,"inject, stat_rewrite",,False,augment
4,symbol_pile_no_grammar,398.0,93,"inject, stat_rewrite",,False,augment
5,bibliographic_catalog_identifier,398.0,339,"inject, stat_rewrite",,False,augment
6,bio_clinical_identifier,398.0,733,"inject, stat_rewrite",,False,augment
7,travel_transport_code,397.0,33,"inject, stat_rewrite",,False,augment
8,standards_compliance_lookup,394.0,58,"inject, stat_rewrite",,False,augment
9,boolean_operator_query,393.0,4304,operator_syntax_rewrite,,False,augment


In [5]:
CELL = "version_pinned_technical"          # two mints — exercises the pipeline
result, frame = loop.demand(CELL)

# grounded() attaches the gold document, but only when the plan CUTS — the cut
# has to keep that document answering, so it reads it (d53d)
parent = loop.grounded(result, frame.iloc[0])
parent[["dataset", "query_id", "query", "surfaces", "bank"]]

parents: dropped 2 row(s) with no query text


dataset                                   msmarco-passage-dev
query_id                                              1040038
query       what is the empirical formula for phosphorus s...
surfaces                                              (1.09,)
bank                                           version_string
dtype: object

In [6]:
# d55: there is no longer ONE instruction per row. A cell becomes a sequence of
# calls — every addition in the first, the cut in a second — and each call's
# targets accumulate, so a later call cannot undo an earlier one.
from augmentation.dispatch import calls_for, targets_of
from composition.cells import CELLS_BY_NAME

for i, call in enumerate(calls_for(result, CELLS_BY_NAME[CELL], parent), 1):
    mints = [s.operator.declaration.operator for s in call.steps]
    print(f"{'=' * 25} CALL {i}  mints={mints}  cuts={call.cuts} {'=' * 25}")
    print(loop.brief(CELL, call.steps, parent))
    print("\nVERIFIED:", targets_of(call.verified, parent).model_dump_json(indent=1))
    print()

========================= CALL 1  mints=['inject']  cuts=False =========================
Weave the exact text '1.09' into the user's search query, on the same topic. The query may narrow — it no longer has to mean exactly what it meant. Insert them character for character, and do not alter them.

The finished query must look like this: A short technical query of three to nine words pinned to an exact dotted version string that must appear verbatim — the kind of precision embeddings actively blur, since adjacent versions differ by the whole point. Vary the job of the surrounding words: sometimes the version alone tags the answer, as in changelogs and release notes; sometimes the accompanying vocabulary — breaking change, migration, incompatibility — must select the right document among the many that mention the same version.


Add nothing beyond what is asked above: no other facts, names, numbers, dates, identifiers, greetings, or politeness phrases — anything extra changes the query's 

In [7]:
from augmentation.pool import GeneratedPool
from augmentation.qrels import AugmentationQrels

In [8]:
throwaway = AugmentationPaths(data_dir="/tmp/smoke")

loop = AugmentationLoop(
    selection, sheet_path=paths.cell_order_sheet, parents=parents,
    pool=GeneratedPool(throwaway), qrels=AugmentationQrels(throwaway),
)
loop.run("version_pinned_technical", n=1)

parents: dropped 2 row(s) with no query text
NOTE: 'inject' is gated by coherence_gate (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.
NOTE: 'stat_rewrite' is gated by declaration_audit (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.


augment:version_pinned_technical:   0%|          | 0/1 [00:02<?, ?row/s, attempted=1, dropped=1]

- 1040038: dropped — failed ['version_string: at least 1 span(s)']
    before: 'what is the empirical formula for phosphorus selenide'
    tried:  'empirical formula phosphorus selenide 1.09 compound'


augment:version_pinned_technical:   0%|          | 0/1 [00:06<?, ?row/s, attempted=2, dropped=2]

- MATH-q-4368: dropped — failed ['version_string: at least 1 span(s)']
    before: 'Problem: Compute $\\tan 45^\\circ$.'
    tried:  'Compute tan 45 degrees trigonometry 1.2'


...

augment:version_pinned_technical:   0%|          | 0/1 [00:21<?, ?row/s, attempted=9, dropped=9]

- PLAIN-1929: dropped — failed ['version_string: at least 1 span(s)']
    before: 'prenatal vitamins'
    tried:  'prenatal vitamins 0.01 breaking changes'


augment:version_pinned_technical: 100%|██████████| 1/1 [00:23<00:00, 23.56s/row, attempted=10, dropped=9]

+ MATH-q-4362 (2 attempt(s))
    before: 'Problem: Compute $\\tan 300^\\circ$.'
    after:  'compute tan 300 degrees version 1.2'
version_pinned_technical: accepted 1/10 attempts (need 1, parents available 4,795, minting ['inject', 'stat_rewrite']) -> /tmp/smoke/augmentation/pool.parquet


,query_id,query,floor,operator,provenance,generated_from,parent_dataset,home_lane,grounding_doc_id,meaning_preserved,answer_key,attempts,credit_gate
0,aug-version_pinned_technical-MATH-q-4362,compute tan 300 degrees version 1.2,version_pinned_technical,inject,doc_grounded,MATH-q-4362,rarb-math,rarb-math,MATH-d-4362,False,minted,2,coherence_gate


In [9]:
loop.run("symbol_pile_no_grammar", n=1)

parents: dropped 8 row(s) with no query text
NOTE: 'inject' is gated by coherence_gate (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.
NOTE: 'stat_rewrite' is gated by declaration_audit (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.


augment:symbol_pile_no_grammar: 100%|██████████| 1/1 [00:06<00:00,  6.29s/row, attempted=1, dropped=0]

+ PLAIN-1485 (1 attempt(s))
    before: 'lard'
    after:  'lard microM nM'
symbol_pile_no_grammar: accepted 1/1 attempts (need 1, parents available 84, minting ['inject', 'stat_rewrite']) -> /tmp/smoke/augmentation/pool.parquet


,query_id,query,floor,operator,provenance,generated_from,parent_dataset,home_lane,grounding_doc_id,meaning_preserved,answer_key,attempts,credit_gate
0,aug-symbol_pile_no_grammar-PLAIN-1485,lard microM nM,symbol_pile_no_grammar,inject,doc_grounded,PLAIN-1485,beir-nfcorpus,beir-nfcorpus,MED-2662,False,minted,1,coherence_gate


In [6]:
from augmentation.campaign import AugmentationCampaign

campaign = AugmentationCampaign(loop)
campaign.plan()

campaign plan — 22 hungry floors, 0 target rows (>= 0 LLM calls):
                           floor  missing operator gate            action  target_rows
        legal_citation_canonical    400.0     None None skip: no operator            0
          datetime_token_present    399.0     None None skip: no operator            0
          single_token_char_blob    398.0     None None skip: no operator            0
     business_temporal_reference    398.0     None None skip: no operator            0
          symbol_pile_no_grammar    398.0     None None skip: no operator            0
bibliographic_catalog_identifier    398.0     None None skip: no operator            0
         bio_clinical_identifier    398.0     None None skip: no operator            0
           travel_transport_code    397.0     None None skip: no operator            0
     standards_compliance_lookup    394.0     None None skip: no operator            0
          boolean_operator_query    393.0     None None skip: no

,floor,missing,operator,gate,action,target_rows
0,legal_citation_canonical,400.0,None,None,skip: no operator,0
1,datetime_token_present,399.0,None,None,skip: no operator,0
2,single_token_char_blob,398.0,None,None,skip: no operator,0
3,business_temporal_reference,398.0,None,None,skip: no operator,0
4,symbol_pile_no_grammar,398.0,None,None,skip: no operator,0
5,bibliographic_catalog_identifier,398.0,None,None,skip: no operator,0
6,bio_clinical_identifier,398.0,None,None,skip: no operator,0
7,travel_transport_code,397.0,None,None,skip: no operator,0
8,standards_compliance_lookup,394.0,None,None,skip: no operator,0
9,boolean_operator_query,393.0,None,None,skip: no operator,0
